# ML-08 — Capstone Modeling: CTR / Engagement Opportunity Scoring

**Lane 4: CTR / Engagement Opportunity Scoring**

Skills loaded: `training-honest-models/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

All claims use careful, observed language (observational / measured / directional / decision-support).  
No client names, domains, URLs, or private queries appear anywhere in this notebook.

**Goal**: train a model that fits Lane 4's question, compare it honestly against the Week-4 rule baseline  
on the **same data, same split, same metric**, then read the errors before believing any score.

---
## 1. Method Choice and Why

**Lane 4 question shape**: *which pages should be reviewed for CTR optimisation first?*  
This is a **ranking / 'which first?' problem** with an observed binary label (`is_low_ctr_for_tier`).  
The honest metric is therefore **precision@K** (of the top-K items the model ranks, how many are true positives?) alongside ROC-AUC for threshold-free discrimination.

**Method ladder (simple → complex):**

| Model | Why considered |
|---|---|
| **Logistic Regression** | Readable coefficients; fastest sanity check; baseline for "does anything beat a linear boundary?" |
| **Decision Tree (depth 3)** | Printable rule; shows the single split structure the data supports; directly comparable to the hand-written baseline |
| **Random Forest** | Captures non-linear interactions; outputs calibrated probabilities for ranking; permutation importance is reliable |

**Why not Gradient Boosting here?**  
The label base rate is 10% and the feature set is only 5 columns — a Random Forest captures the signal without overfitting on this scale. Gradient Boosting adds tuning overhead (learning rate, n_estimators, depth) without a clear payoff when the baseline itself scores near-perfectly.

**The honest prior**: the Week-4 rule baseline already exploits the CTR-gap signal directly (score = `ctr_gap × log1p(impressions)`). Any model trained on a feature set that *excludes* raw CTR faces a structural ceiling — it must infer the gap indirectly from position and engagement signals. If the model cannot beat the baseline, **that is the finding**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA_PATH)

# ── Reproduce w04 working set (same filter as baseline) ───────────────────
working = df[
    (df['avg_position'] > 0) &
    (df['impressions_90d'] >= 100) &
    (df['position_tier'] != 'no_data')
].copy()

# ── Label (same as w03 contract) ──────────────────────────────────────────
tier_p25 = working.groupby('position_tier')['ctr'].quantile(0.25)
working['tier_p25_ctr'] = working['position_tier'].map(tier_p25)
working['is_low_ctr_for_tier'] = (working['ctr'] < working['tier_p25_ctr']).astype(int)

# ── Five honest features from w03 (no CTR reconstruction path) ───────────
working['log_imp_month']             = np.log1p(working['impressions_90d'])
working['avg_pos_month']             = working['avg_position']
working['ga4_eng_rate']              = working['engagement_rate']
working['pct_days_with_impressions'] = working['days_with_impressions'] / 90 * 100
working['days_since_update']         = working['days_since_last_update']

FEATURE_COLS = [
    'log_imp_month', 'avg_pos_month', 'ga4_eng_rate',
    'pct_days_with_impressions', 'days_since_update'
]

model_df = working.dropna(subset=FEATURE_COLS + ['is_low_ctr_for_tier']).copy().reset_index(drop=True)
X = model_df[FEATURE_COLS]
y = model_df['is_low_ctr_for_tier']
groups = model_df['client_id']

print(f'Working set   : {len(model_df):,} rows')
print(f'Clients       : {groups.nunique()} (used as CV groups)')
print(f'Label base rate: {y.mean():.3f} ({100*y.mean():.1f}% positives = {y.sum():,} rows)')
print(f'Features      : {FEATURE_COLS}')
print(f'Random seed   : {RANDOM_SEED}')

Working set   : 22,006 rows
Clients       : 30 (used as CV groups)
Label base rate: 0.100 (10.0% positives = 2,202 rows)
Features      : ['log_imp_month', 'avg_pos_month', 'ga4_eng_rate', 'pct_days_with_impressions', 'days_since_update']
Random seed   : 42


---
## 2. Split Design

**Why grouped by client:**  
The 30 clients in the dataset have different sizes, query niches, and CTR baselines. A random row split
would leak client-specific patterns (e.g. a client with structurally low CTR across all pages) from train
into test — the model would effectively memorise the client rather than learn the underlying signal.

**Design:** 5-fold **GroupKFold** with `client_id` as the grouping key.  
Each fold uses ~24 clients for training and ~6 clients for testing, with no client appearing in both.  
This is a conservative estimate of out-of-client generalisation — the real deployment scenario.

**Note on time:** the starter CSV is a single 90-day snapshot without a daily `report_date` column;
a pure time-ordered split is not possible here. GroupKFold by client is the correct conservative
alternative — it tests whether the learned pattern generalises across clients, not just across pages
within the same client.

In [2]:
gkf = GroupKFold(n_splits=5)

# Show the fold structure
print('GroupKFold split structure (5 folds, grouped by client_id):')
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups), 1):
    tr_clients = groups.iloc[tr_idx].nunique()
    te_clients = groups.iloc[te_idx].nunique()
    te_base    = y.iloc[te_idx].mean()
    print(f'  Fold {fold}: {len(tr_idx):>6,} train rows ({tr_clients} clients) | '
          f'{len(te_idx):>6,} test rows ({te_clients} clients) | '
          f'test base rate = {te_base:.3f}')
print()
print('No client appears in both train and test within any fold.')

GroupKFold split structure (5 folds, grouped by client_id):
  Fold 1: 15,427 train rows (29 clients) |  6,579 test rows (1 clients) | test base rate = 0.117
  Fold 2: 18,148 train rows (25 clients) |  3,858 test rows (5 clients) | test base rate = 0.050
  Fold 3: 18,150 train rows (23 clients) |  3,856 test rows (7 clients) | test base rate = 0.121
  Fold 4: 18,150 train rows (22 clients) |  3,856 test rows (8 clients) | test base rate = 0.082
  Fold 5: 18,149 train rows (21 clients) |  3,857 test rows (9 clients) | test base rate = 0.118

No client appears in both train and test within any fold.


---
## 3. Train + Compare vs My Baseline

**Week-4 rule baseline reminder:**  
`score = (tier_median_ctr − page_ctr) × log1p(impressions_90d)`  
Tier medians computed on the **training fold** only — same CV loop as the models.

**Metric definitions:**
- **ROC-AUC** — threshold-free discrimination; 0.5 = random, 1.0 = perfect
- **Avg Precision** — area under the precision-recall curve; accounts for class imbalance
- **Precision@20 / @50** — of the top 20 / top 50 flagged pages, what fraction are true positives? This is the operational metric for the CTR review queue.

In [3]:
def cv_scores(model, X, y, groups, cv, name):
    """Run grouped CV and return mean scores across folds."""
    aucs, aps, pks20, pks50 = [], [], [], []
    for tr_idx, te_idx in cv.split(X, y, groups):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, probs))
        aps.append(average_precision_score(y_te, probs))
        order = np.argsort(-probs)
        pks20.append(y_te.iloc[order[:20]].mean())
        pks50.append(y_te.iloc[order[:50]].mean())
    return {
        'Model': name,
        'ROC-AUC': round(np.mean(aucs), 3),
        'AUC±': round(np.std(aucs), 3),
        'Avg Precision': round(np.mean(aps), 3),
        'P@20': round(np.mean(pks20), 3),
        'P@50': round(np.mean(pks50), 3),
    }


def baseline_cv_scores(X, y, groups, cv, model_df):
    """Week-4 rule baseline evaluated in the same CV loop."""
    aucs, aps, pks20, pks50 = [], [], [], []
    for tr_idx, te_idx in cv.split(X, y, groups):
        # Tier medians computed from TRAIN fold only (no leakage)
        train_w = model_df.iloc[tr_idx]
        tier_med = train_w.groupby('position_tier')['ctr'].median()
        test_w = model_df.iloc[te_idx].copy()
        test_w['tier_med_ctr'] = test_w['position_tier'].map(tier_med).fillna(0)
        score = (test_w['tier_med_ctr'] - test_w['ctr']) * np.log1p(test_w['impressions_90d'])
        y_te = y.iloc[te_idx]
        aucs.append(roc_auc_score(y_te, score))
        aps.append(average_precision_score(y_te, score))
        order = np.argsort(-score.values)
        pks20.append(y_te.iloc[order[:20]].mean())
        pks50.append(y_te.iloc[order[:50]].mean())
    return {
        'Model': 'W4 Rule Baseline',
        'ROC-AUC': round(np.mean(aucs), 3),
        'AUC±': round(np.std(aucs), 3),
        'Avg Precision': round(np.mean(aps), 3),
        'P@20': round(np.mean(pks20), 3),
        'P@50': round(np.mean(pks50), 3),
    }


print('Running 5-fold GroupKFold CV for all models...')
print('(This may take ~60 seconds for the Random Forest)')
print()

Running 5-fold GroupKFold CV for all models...
(This may take ~60 seconds for the Random Forest)



In [4]:
# ── Week-4 rule baseline ──────────────────────────────────────────────────
base_row = baseline_cv_scores(X, y, groups, gkf, model_df)
print(f"W4 Baseline: AUC={base_row['ROC-AUC']}, P@20={base_row['P@20']}, P@50={base_row['P@50']}")

# ── Logistic Regression ──────────────────────────────────────────────────
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', C=1.0,
                               random_state=RANDOM_SEED, max_iter=1000))
])
lr_row = cv_scores(lr_pipe, X, y, groups, gkf, 'Logistic Regression')
print(f"LR:          AUC={lr_row['ROC-AUC']}, P@20={lr_row['P@20']}, P@50={lr_row['P@50']}")

# ── Decision Tree (depth 3, readable) ────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50,
                             class_weight='balanced', random_state=RANDOM_SEED)
dt_row = cv_scores(dt, X, y, groups, gkf, 'Decision Tree (depth 3)')
print(f"DT depth-3:  AUC={dt_row['ROC-AUC']}, P@20={dt_row['P@20']}, P@50={dt_row['P@50']}")

# ── Random Forest ────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                             class_weight='balanced',
                             random_state=RANDOM_SEED, n_jobs=-1)
rf_row = cv_scores(rf, X, y, groups, gkf, 'Random Forest')
print(f"RF:          AUC={rf_row['ROC-AUC']}, P@20={rf_row['P@20']}, P@50={rf_row['P@50']}")

W4 Baseline: AUC=0.988, P@20=1.0, P@50=0.996
LR:          AUC=0.868, P@20=0.65, P@50=0.632


DT depth-3:  AUC=0.909, P@20=0.45, P@50=0.48


RF:          AUC=0.922, P@20=0.74, P@50=0.684


In [5]:
# ── Comparison table ──────────────────────────────────────────────────────
results = pd.DataFrame([base_row, lr_row, dt_row, rf_row])

print('=' * 75)
print('MODEL VS BASELINE COMPARISON — 5-fold GroupKFold by client_id')
print(f'Base rate: {y.mean():.3f} ({100*y.mean():.1f}% positives)')
print('=' * 75)
print(results.to_string(index=False))
print()
print('Columns:')
print('  ROC-AUC      : area under ROC curve (threshold-free; ± = std across folds)')
print('  Avg Precision: area under precision-recall curve (imbalance-aware)')
print('  P@20 / P@50  : precision@K -- fraction of top-K flagged pages that are true positives')
print()
print('READING:')
print('  The W4 rule baseline achieves P@20=1.0 and P@50=0.996 because the rule directly')
print('  exploits the CTR gap (page_ctr vs tier_median_ctr) -- which is monotonically related')
print('  to the label. The 5-feature model set EXCLUDES raw CTR (leakage), so any learned model')
print('  must infer the gap from position and engagement signals alone.')
print()
print('  Random Forest AUC=0.922 shows real discriminative signal in the safe features,')
print('  but cannot match precision@K because it has no direct access to the CTR gap.')
print('  This is the expected and honest outcome: the rule is hard to beat in its own domain.')

MODEL VS BASELINE COMPARISON — 5-fold GroupKFold by client_id
Base rate: 0.100 (10.0% positives)
                  Model  ROC-AUC  AUC±  Avg Precision  P@20  P@50
       W4 Rule Baseline    0.988 0.010          0.917  1.00 0.996
    Logistic Regression    0.868 0.075          0.407  0.65 0.632
Decision Tree (depth 3)    0.909 0.043          0.422  0.45 0.480
          Random Forest    0.922 0.035          0.519  0.74 0.684

Columns:
  ROC-AUC      : area under ROC curve (threshold-free; ± = std across folds)
  Avg Precision: area under precision-recall curve (imbalance-aware)
  P@20 / P@50  : precision@K -- fraction of top-K flagged pages that are true positives

READING:
  The W4 rule baseline achieves P@20=1.0 and P@50=0.996 because the rule directly
  exploits the CTR gap (page_ctr vs tier_median_ctr) -- which is monotonically related
  to the label. The 5-feature model set EXCLUDES raw CTR (leakage), so any learned model
  must infer the gap from position and engagement signals alo

---
## 4. Errors and Interpretation

Fitting the Random Forest on a held-out client group for error analysis.

In [6]:
# Final fit on held-out client group
# Use the 7 largest clients (~25% of rows) as a held-out test set for error analysis
client_sizes = model_df.groupby('client_id').size().sort_values()
n_test_clients = max(1, int(len(client_sizes) * 0.25))
test_clients = set(client_sizes.index[-n_test_clients:])

tr_mask = ~model_df['client_id'].isin(test_clients)
te_mask =  model_df['client_id'].isin(test_clients)

X_tr, X_te = X[tr_mask].reset_index(drop=True), X[te_mask].reset_index(drop=True)
y_tr, y_te = y[tr_mask].reset_index(drop=True), y[te_mask].reset_index(drop=True)

rf_final = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                   class_weight='balanced',
                                   random_state=RANDOM_SEED, n_jobs=-1)
rf_final.fit(X_tr, y_tr)

y_prob = rf_final.predict_proba(X_te)[:, 1]
y_pred = rf_final.predict(X_te)

print(f'Held-out test set: {te_mask.sum():,} rows | {len(test_clients)} clients')
print(f'Test base rate: {y_te.mean():.3f} ({100*y_te.mean():.1f}%)')
print(f'Test ROC-AUC: {roc_auc_score(y_te, y_prob):.3f}')
print(f'Test Avg Precision: {average_precision_score(y_te, y_prob):.3f}')
order_te = np.argsort(-y_prob)
print(f'Test P@20: {y_te.iloc[order_te[:20]].mean():.3f}')
print(f'Test P@50: {y_te.iloc[order_te[:50]].mean():.3f}')

Held-out test set: 17,337 rows | 7 clients
Test base rate: 0.093 (9.3%)
Test ROC-AUC: 0.912
Test Avg Precision: 0.449
Test P@20: 0.600
Test P@50: 0.580


In [7]:
# Permutation importance on the held-out test set
perm = permutation_importance(
    rf_final, X_te, y_te,
    n_repeats=20, random_state=RANDOM_SEED, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_mean': perm.importances_mean.round(4),
    'importance_std':  perm.importances_std.round(4),
}).sort_values('importance_mean', ascending=False)

print('Permutation importance (drop in ROC-AUC when feature is shuffled):')
print(perm_df.to_string(index=False))
print()
print('Interpretation:')
print('  avg_pos_month (+0.367): by far the strongest signal.')
print('    Position within search results is the most reliable proxy for CTR tier:')
print('    pages deeper than position 10 structurally cannot be in the top-CTR tier.')
print()
print('  ga4_eng_rate (+0.031): second-most important.')
print('    Pages with very low engagement rates (sessions that bounce quickly) often have')
print('    low CTR too -- the SERP snippet under-promises relative to page content, or')
print('    query intent is misdirected.')
print()
print('  log_imp_month (+0.012): modest but positive.')
print('    High-impression pages appear more often in CTR-poor positions;')
print('    impression volume gives the model scale context.')
print()
print('  days_since_update (+0.001): near-zero.')
print('    Content staleness is a weak proxy in this label -- the CTR-gap label does')
print('    not require recency, and stale pages span all CTR tiers.')

Permutation importance (drop in ROC-AUC when feature is shuffled):
                  feature  importance_mean  importance_std
            avg_pos_month           0.3673          0.0071
             ga4_eng_rate           0.0308          0.0020
            log_imp_month           0.0115          0.0014
pct_days_with_impressions           0.0090          0.0010
        days_since_update           0.0005          0.0007

Interpretation:
  avg_pos_month (+0.367): by far the strongest signal.
    Position within search results is the most reliable proxy for CTR tier:
    pages deeper than position 10 structurally cannot be in the top-CTR tier.

  ga4_eng_rate (+0.031): second-most important.
    Pages with very low engagement rates (sessions that bounce quickly) often have
    low CTR too -- the SERP snippet under-promises relative to page content, or
    query intent is misdirected.

  log_imp_month (+0.012): modest but positive.
    High-impression pages appear more often in CTR-poor posi

In [8]:
# Decision tree depth-3: the readable version of what RF learned
dt_final = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=50,
    class_weight='balanced', random_state=RANDOM_SEED
)
dt_final.fit(X_tr, y_tr)

print('Decision Tree (depth 3) — human-readable rule:')
print()
print(export_text(dt_final, feature_names=FEATURE_COLS))
print()
print('Plain-English reading:')
print('  If avg_position <= 10 (page_1 or better) AND low impressions AND low engagement:')
print('  => likely low-CTR-for-tier (positive label)')
print('  If avg_position <= 10 AND high impressions AND decent engagement:')
print('  => likely NOT low CTR (negative)')
print('  If avg_position > 10 (striking/deep):')
print('  => likely NOT low CTR for tier (these tiers have near-zero tier p25 CTR,')
print('     so few pages fall below it)')

Decision Tree (depth 3) — human-readable rule:

|--- avg_pos_month <= 10.05
|   |--- log_imp_month <= 7.48
|   |   |--- ga4_eng_rate <= 1.19
|   |   |   |--- class: 1
|   |   |--- ga4_eng_rate >  1.19
|   |   |   |--- class: 1
|   |--- log_imp_month >  7.48
|   |   |--- ga4_eng_rate <= 0.12
|   |   |   |--- class: 1
|   |   |--- ga4_eng_rate >  0.12
|   |   |   |--- class: 0
|--- avg_pos_month >  10.05
|   |--- class: 0


Plain-English reading:
  If avg_position <= 10 (page_1 or better) AND low impressions AND low engagement:
  => likely low-CTR-for-tier (positive label)
  If avg_position <= 10 AND high impressions AND decent engagement:
  => likely NOT low CTR (negative)
  If avg_position > 10 (striking/deep):
  => likely NOT low CTR for tier (these tiers have near-zero tier p25 CTR,
     so few pages fall below it)


In [9]:
# Error analysis
test_df = model_df[te_mask].copy().reset_index(drop=True)
test_df['prob'] = y_prob
test_df['pred'] = y_pred

fp = test_df[(test_df['pred'] == 1) & (test_df['is_low_ctr_for_tier'] == 0)].copy()
fn = test_df[(test_df['pred'] == 0) & (test_df['is_low_ctr_for_tier'] == 1)].copy()
tp = test_df[(test_df['pred'] == 1) & (test_df['is_low_ctr_for_tier'] == 1)].copy()

print(f'Confusion matrix (held-out test set):')
print(f'  True Positives  (TP): {len(tp):>5,}  -- correctly flagged for CTR review')
print(f'  False Positives (FP): {len(fp):>5,}  -- flagged but CTR is actually fine')
print(f'  False Negatives (FN): {len(fn):>5,}  -- missed low-CTR pages')
print(f'  True Negatives  (TN): {len(test_df)-len(tp)-len(fp)-len(fn):>5,}')
print()

print('FP profile (flagged as low CTR, but actually at/above tier p25):')
print(f'  Count: {len(fp):,}')
print(f'  Median CTR      : {fp["ctr"].median():.3f}  (tier p25 is ~0.09 for page_1)')
print(f'  Median position : {fp["avg_pos_month"].median():.1f}')
print(f'  Top tier        : {fp["position_tier"].value_counts().index[0]} ({fp["position_tier"].value_counts().iloc[0]:,} rows)')
print(f'  Why hard: page_1 pages with low-to-moderate CTR (e.g. 0.27%) sit near the p25')
print(f'  threshold (0.09%). A small model uncertainty pushes them into the positive class.')
print()

print('FN profile (low CTR missed by the model):')
print(f'  Count: {len(fn):,}')
print(f'  Median impressions: {fn["impressions_90d"].median():,.0f}')
print(f'  Median CTR        : {fn["ctr"].median():.3f}  (below tier p25)')
print(f'  Median position   : {fn["avg_pos_month"].median():.1f}')
print(f'  Why hard: high-impression pages at page_1 depths with CTR just below tier p25')
print(f'  look similar to TP pages in the feature space -- the model resolves the ambiguity wrong.')
print(f'  These are the most actionable misses because they represent genuine CTR opportunities')
print(f'  on high-visibility pages.')

Confusion matrix (held-out test set):
  True Positives  (TP): 1,061  -- correctly flagged for CTR review
  False Positives (FP): 1,495  -- flagged but CTR is actually fine
  False Negatives (FN):   558  -- missed low-CTR pages
  True Negatives  (TN): 14,223

FP profile (flagged as low CTR, but actually at/above tier p25):
  Count: 1,495
  Median CTR      : 0.270  (tier p25 is ~0.09 for page_1)
  Median position : 7.5
  Top tier        : page_1 (1,391 rows)
  Why hard: page_1 pages with low-to-moderate CTR (e.g. 0.27%) sit near the p25
  threshold (0.09%). A small model uncertainty pushes them into the positive class.

FN profile (low CTR missed by the model):
  Count: 558
  Median impressions: 4,878
  Median CTR        : 0.050  (below tier p25)
  Median position   : 6.6
  Why hard: high-impression pages at page_1 depths with CTR just below tier p25
  look similar to TP pages in the feature space -- the model resolves the ambiguity wrong.
  These are the most actionable misses because t

In [10]:
# Three concrete hard cases
print('Three concrete hard cases (FN — missed, should have been flagged):')
print()
hard_fn = fn.sort_values('impressions_90d', ascending=False).head(3)
for i, (_, row) in enumerate(hard_fn.iterrows(), 1):
    print(f'  Case {i}:')
    print(f'    Impressions   : {row["impressions_90d"]:,}')
    print(f'    Avg position  : {row["avg_pos_month"]:.1f}')
    print(f'    CTR           : {row["ctr"]:.3f}%  (tier p25 = {row["tier_p25_ctr"]:.3f}%)')
    print(f'    Engagement    : {row["ga4_eng_rate"]:.2f}%')
    print(f'    Model prob    : {row["prob"]:.3f}  (predicted: NOT low-CTR)')
    print(f'    Why hard      : High impressions + decent engagement rate mislead the model')
    print(f'                    into predicting "above tier CTR" even though CTR is below p25.')
    print(f'                    The rule baseline catches this because it directly computes the gap.')
    print()

Three concrete hard cases (FN — missed, should have been flagged):

  Case 1:
    Impressions   : 295,097
    Avg position  : 7.3
    CTR           : 0.050%  (tier p25 = 0.090%)
    Engagement    : 1.68%
    Model prob    : 0.181  (predicted: NOT low-CTR)
    Why hard      : High impressions + decent engagement rate mislead the model
                    into predicting "above tier CTR" even though CTR is below p25.
                    The rule baseline catches this because it directly computes the gap.

  Case 2:
    Impressions   : 223,271
    Avg position  : 7.8
    CTR           : 0.030%  (tier p25 = 0.090%)
    Engagement    : 3.45%
    Model prob    : 0.128  (predicted: NOT low-CTR)
    Why hard      : High impressions + decent engagement rate mislead the model
                    into predicting "above tier CTR" even though CTR is below p25.
                    The rule baseline catches this because it directly computes the gap.

  Case 3:
    Impressions   : 147,670
    Avg posi

In [11]:
# Final summary table
print('=' * 75)
print('FINAL SUMMARY: MODEL vs BASELINE')
print(f'Data: {len(model_df):,} rows | Label base rate: {y.mean():.3f} | Split: 5-fold GroupKFold by client')
print('=' * 75)
print(results.to_string(index=False))
print()
print('Conclusion (3 sentences):')
print('  The W4 rule baseline achieves near-perfect precision@K because its score directly')
print('  encodes the CTR gap -- the same signal the label is defined on. The Random Forest')
print('  (AUC=0.922) and Decision Tree (AUC=0.909) learn real signal from position and')
print('  engagement features alone, but cannot match the rule where it matters most: the')
print('  top of the ranked queue. The most informative finding from the model is the')
print('  permutation importance: avg_pos_month (+0.367) dominates, confirming that position')
print('  tier is the strongest indirect proxy for CTR potential -- matching the rule logic.')

FINAL SUMMARY: MODEL vs BASELINE
Data: 22,006 rows | Label base rate: 0.100 | Split: 5-fold GroupKFold by client
                  Model  ROC-AUC  AUC±  Avg Precision  P@20  P@50
       W4 Rule Baseline    0.988 0.010          0.917  1.00 0.996
    Logistic Regression    0.868 0.075          0.407  0.65 0.632
Decision Tree (depth 3)    0.909 0.043          0.422  0.45 0.480
          Random Forest    0.922 0.035          0.519  0.74 0.684

Conclusion (3 sentences):
  The W4 rule baseline achieves near-perfect precision@K because its score directly
  encodes the CTR gap -- the same signal the label is defined on. The Random Forest
  (AUC=0.922) and Decision Tree (AUC=0.909) learn real signal from position and
  engagement features alone, but cannot match the rule where it matters most: the
  top of the ranked queue. The most informative finding from the model is the
  permutation importance: avg_pos_month (+0.367) dominates, confirming that position
  tier is the strongest indirect prox

---
## 5. Self-Check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, domains, URLs, or private queries appear anywhere
- [x] All claims use careful words: observed, measured, directional, decision-support
- [x] Method choice explained (LR → DT → RF ladder; GBM not used — justified)
- [x] Split design: GroupKFold by client_id — reason explained (prevent client memorisation)
- [x] Comparison table: W4 baseline + LR + DT + RF on same data, same split, same metrics
- [x] Baseline appears in the same notebook run as the models (not copy-pasted)
- [x] Feature importances: permutation importance + human explanation per feature
- [x] Error analysis: FP and FN profiles + 3 concrete hard cases with reasoning
- [x] Honest conclusion: model does not beat baseline — that IS the finding
- [x] Random seed fixed (42) and stated; reproducible
- [x] Committed to repo under `work/notebooks/` — repo URL submitted on the card

**Done.**